# Iterative Title Generation and Optimization

This notebook uses the trained numeric inference models and qualitative LLM analysis to iteratively improve video titles for specific channels.

In [ ]:
# Setup and Dependencies
!pip install -q -U google-generativeai sentence-transformers scikit-learn pandas numpy statsmodels

import os
import json
import time
import numpy as np
import pandas as pd
from google.colab import drive, userdata
import google.generativeai as genai
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

drive.mount('/content/drive')

GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
MODEL_NAME = 'gemini-3.1-flash-lite-latest'
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'

BASE_PATH = '/content/drive/MyDrive/numeric_inference_outputs/'
EVAL_DATA_PATH = os.path.join(BASE_PATH, 'top_significant_channels_eval.json')
LLM_RESULTS_PATH = os.path.join(BASE_PATH, 'llm_analysis_results.json')
EMBEDDING_CACHE_PATH = os.path.join(BASE_PATH, 'video_title_embeddings_latest.json')
TRAIN_DATA_PATH = os.path.join(BASE_PATH, 'train_structured_latest.json')

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

In [ ]:
# Data Loading
with open(EVAL_DATA_PATH, 'r') as f:
    eval_dataset = json.load(f)

with open(LLM_RESULTS_PATH, 'r') as f:
    llm_analysis = json.load(f)

with open(EMBEDDING_CACHE_PATH, 'r') as f:
    embedding_cache = json.load(f)

with open(TRAIN_DATA_PATH, 'r') as f:
    train_data = json.load(f)

print("Data loaded successfully.")

In [ ]:
# Reconstruct global PCA from training data
all_train_embeddings = []
for channel in train_data:
    for video in channel['videos']:
        all_train_embeddings.append(embedding_cache[video['title']])

X_train = np.array(all_train_embeddings)
pca = PCA(n_components=15, random_state=42)
pca.fit(X_train)

print(f"Reconstructed PCA with {pca.n_components_} components.")

In [ ]:
# Utility functions for prediction
def predict_views(titles, channel_id, channel_models_params):
    """
    Predict log-views for a list of titles.
    """
    # 1. Embed titles
    embs = embedding_model.encode(titles)
    
    # 2. Project to PCA
    reduced = pca.transform(embs)
    
    # 3. Add constant
    X = np.column_stack([np.ones(len(reduced)), reduced])
    
    # 4. Predict using coefficients
    params = channel_models_params[channel_id]
    coeffs = np.array(params['coefficients'])
    intercept = params['intercept']
    
    # coefficients in eval file usually include the intercept as first element if using sm.OLS
    # but let's check structure from params
    all_coeffs = np.array(params['all_params'])
    predictions = X @ all_coeffs
    
    return predictions

In [ ]:
# Configuration for target channels
TARGET_CHANNELS_REQUESTED = ["20VC with Harry Stebbings", "A16z"]
target_channel_data = []
channel_models_params = {}

# Case-insensitive matching
target_channels_lower = [c.lower() for c in TARGET_CHANNELS_REQUESTED]

for channel in eval_dataset:
    if channel['channel_name'].lower() in target_channels_lower:
        target_channel_data.append(channel)
        channel_models_params[channel['channel_id']] = {
            'all_params': channel['model']['all_params'],
            'coefficients': channel['model']['coefficients'],
            'intercept': channel['model']['intercept']
        }

print(f"Targeted {len(target_channel_data)} channels.")

In [ ]:
def get_gemini_response(prompt):
    model = genai.GenerativeModel(MODEL_NAME)
    for _ in range(3):
        try:
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            print(f"Error: {e}. Retrying...")
            time.sleep(5)
    return ""

def parse_titles(text):
    """Extract 10 titles from LLM response."""
    lines = text.strip().split('\n')
    titles = []
    for line in lines:
        line = line.strip()
        if not line: continue
        # Strip leading numbers or bullets
        if line[0].isdigit() or line[0] in ['-', '*']:
            parts = line.split('. ', 1) if '.' in line else line.split(' ', 1)
            if len(parts) > 1:
                titles.append(parts[1].strip().strip('"'))
            else:
                titles.append(line.strip('-* ').strip('"'))
        else:
            titles.append(line.strip('"'))
        if len(titles) == 10: break
    return titles[:10]

In [ ]:
# Iterative Optimization Loop
results = []

for channel in target_channel_data:
    cid = channel['channel_id']
    cname = channel['channel_name']
    
    global_desc = llm_analysis['global_performance_descriptions'].get(cid, "")
    dim_analysis = llm_analysis['channel_significant_dimension_analysis'].get(cid, "")
    
    # Select 10 test videos
    test_vids = channel['test_videos'][:10]
    
    for vid in test_vids:
        base_title = vid['title']
        base_pred = np.log1p(vid['predicted_views'])
        
        print(f"\nOptimizing title for {cname}: '{base_title}'")
        
        history = [] # To store (titles, predictions)
        
        for i in range(5):
            print(f"  Iteration {i+1}...")
            
            # Build Prompt
            prompt = f"""You are an expert YouTube title strategist for the channel '{cname}'.
Channel success drivers:
{global_desc}

Semantic performance analysis:
{dim_analysis}

Original Title: {base_title}
Current best predicted performance (log-views): {max([base_pred] + [max(p) for t, p in history]) if history else base_pred:.4f}
"""
            if history:
                # Include top 3 and bottom 3 from previous iterations for feedback
                all_prev = []
                for t_list, p_list in history:
                    for t, p in zip(t_list, p_list):
                        all_prev.append((t, p))
                all_prev.sort(key=lambda x: x[1], reverse=True)
                
                prompt += "\nPrevious suggestions feedback:\n"
                prompt += "Top performing suggestions:\n"
                for t, p in all_prev[:3]:
                    prompt += f"- {t} (Score: {p:.4f})\n"
                
                prompt += "Lower performing suggestions (Avoid these patterns):\n"
                for t, p in all_prev[-3:]:
                    prompt += f"- {t} (Score: {p:.4f})\n"
            
            prompt += "\nTask: Generate 10 new, improved titles for this video that will maximize views based on the channel's success drivers and semantic analysis. Return ONLY the 10 titles, one per line."
            
            llm_text = get_gemini_response(prompt)
            new_titles = parse_titles(llm_text)
            
            if len(new_titles) < 10:
                print(f"    Warning: Only got {len(new_titles)} titles.")
            
            preds = predict_views(new_titles, cid, channel_models_params)
            history.append((new_titles, preds))
            
            # Identify best this round
            best_idx = np.argmax(preds)
            if preds[best_idx] > base_pred:
                print(f"    Best this round: '{new_titles[best_idx]}' (+{preds[best_idx] - base_pred:.4f} log-views)")
            else:
                print(f"    Best this round: '{new_titles[best_idx]}' ({preds[best_idx] - base_pred:.4f} log-views)")

        # Collect final results for this video
        all_optimized = []
        for t_list, p_list in history:
            for t, p in zip(t_list, p_list):
                all_optimized.append((t, p))
        
        all_optimized.sort(key=lambda x: x[1], reverse=True)
        best_title, best_score = all_optimized[0]
        
        results.append({
            'channel': cname,
            'original_title': base_title,
            'original_score': base_pred,
            'best_optimized_title': best_title,
            'best_optimized_score': best_score,
            'improvement': best_score - base_pred,
            'improvement_pct': (np.expm1(best_score) - np.expm1(base_pred)) / np.expm1(base_pred) * 100 if np.expm1(base_pred) > 0 else 0
        })

print("\nOptimization complete.")

## Final Report

In [ ]:
df_results = pd.DataFrame(results)

# Use the matched channel names for reporting
target_channel_names = [c['channel_name'] for c in target_channel_data]
for channel in target_channel_names:
    print(f"\n=== {channel} Summary ===")
    chan_df = df_results[df_results['channel'] == channel]
    for _, row in chan_df.iterrows():
        print(f"Original: {row['original_title']}")
        print(f"Best:     {row['best_optimized_title']}")
        print(f"Expected Improvement: {row['improvement']:.4f} log-views ({row['improvement_pct']:.2f}%)")
        print("-" * 20)

print(f"\nAverage Improvement across all videos: {df_results['improvement'].mean():.4f} log-views")